# WeShop-UI Try-On (Kaggle + T4 + Gradio Link)

This notebook is designed for **Kaggle**.

1. In Kaggle: **Settings -> Accelerator -> GPU (T4 x2 / T4)**
2. (Recommended) Enable Internet if cloning from GitHub
3. Run all cells; the last cell prints a public **Gradio** URL.


In [ ]:
# 1) Verify GPU (should show Tesla T4 on Kaggle when GPU is enabled)
!nvidia-smi

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 2) Clone WeShop-UI and install dependencies
# If Kaggle asks, make sure Internet is ON in notebook settings.

import os, subprocess, sys

REPO_URL = 'https://github.com/weshopai/WeShop-UI.git'
REPO_DIR = '/kaggle/working/WeShop-UI'

if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)

# Upgrade pip first
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'pip', 'setuptools', 'wheel'])

# Install project dependencies if files exist
if os.path.exists('requirements.txt'):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])

if os.path.exists('pyproject.toml'):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '.'])

# Ensure Gradio exists
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'gradio'])

print('Setup complete. Current dir:', os.getcwd())


In [ ]:
# 3) Auto-detect and launch the repo's Gradio app with public share link
# This cell tries common entrypoints first, then scans for `launch(share=True)`.

import os, re, subprocess, sys

candidates = [
    'app.py',
    'webui.py',
    'gradio_app.py',
    'demo.py',
    'run.py',
    'main.py',
]

entrypoint = None
for c in candidates:
    if os.path.exists(c):
        entrypoint = c
        break

if entrypoint is None:
    py_files = []
    for root, _, files in os.walk('.'):
        for f in files:
            if f.endswith('.py'):
                py_files.append(os.path.join(root, f))

    for f in py_files:
        try:
            txt = open(f, 'r', encoding='utf-8', errors='ignore').read()
        except Exception:
            continue
        if 'gradio' in txt.lower() and ('launch(' in txt or 'Blocks(' in txt or 'Interface(' in txt):
            entrypoint = f
            break

if entrypoint is None:
    raise RuntimeError('Could not find Gradio entrypoint automatically. Open the repo and set entrypoint manually.')

print('Using entrypoint:', entrypoint)

# Create a lightweight wrapper to force share=True when possible
wrapper = '''
import runpy

# Run the target module/script.
# If the script already calls demo.launch(...), Gradio should print a public URL when share=True is set there.
runpy.run_path("%s", run_name="__main__")
''' % entrypoint.replace('\\','/')

with open('/kaggle/working/_launch_weshop.py', 'w') as f:
    f.write(wrapper)

print('Starting app... watch logs for: https://*.gradio.live')
print('If no public URL appears, edit the project launch call to include: share=True')

subprocess.run([sys.executable, '/kaggle/working/_launch_weshop.py'], check=False)


## If share link does not appear

- Open the detected entrypoint file and find `launch(...)`.
- Change it to include `share=True`, for example:

```python
demo.launch(server_name='0.0.0.0', server_port=7860, share=True)
```

Then rerun the launch cell.
